# Crack Detection — Vision Transformer (SegFormer) Segmentation

Trains a transformer-based segmentation model (SegFormer / MiT backbone) on **two combined datasets**:

1. **CrackSeg9k** — https://www.kaggle.com/datasets/lakshaymiddha/crack-segmentation-dataset
2. **CrackVision12K** — https://www.kaggle.com/datasets/vangiap/crackvision12k

Both are treated as binary segmentation problems (crack pixel vs background) and merged into one training set so the model generalizes across surface types (concrete, pavement, walls, etc).

At the end there's an **upload widget** — drop in any image and the notebook will run inference and show the predicted crack mask overlaid on your image.

**Environment**: written for running locally in Jupyter (Notebook or Lab) on your own machine. Tuned for a 4GB-class laptop GPU (e.g. RTX 3050 4GB).

**Resumable training**: the training loop (Section 7) checkpoints to disk after every epoch. If you interrupt it (Kernel → Interrupt, closing Jupyter, laptop sleeps, power cut, whatever) just re-run that cell later — it picks up right after the last completed epoch instead of starting over. See Section 7's markdown for details. You can also load any saved checkpoint independently at any time (Section 9) to test the model without re-running the data pipeline.

**Before running:**
1. Create a virtual env and install Jupyter if you haven't: `python -m venv venv && source venv/bin/activate` (Windows: `venv\Scripts\activate`), then `pip install jupyterlab`.
2. Get a Kaggle API token: kaggle.com → Account → **Create New API Token** → downloads `kaggle.json`.
3. Place it at:
   - Linux/Mac: `~/.kaggle/kaggle.json`
   - Windows: `C:\Users\<you>\.kaggle\kaggle.json`
4. If you have an NVIDIA GPU, make sure you have a CUDA-enabled PyTorch installed (`pip install torch --index-url https://download.pytorch.org/whl/cu121` — pick the CUDA version matching your driver). Without a GPU this will train, just slowly — see the note on batch size / image size below.

> ⚠️ Folder structures inside Kaggle datasets change between versions. Section 2 auto-discovers image/mask folder pairs, but **check the printed paths after running it** and adjust `DATASET_ROOTS` if something looks wrong.


## 1. Setup & Dependencies

In [ ]:
# If you're in a fresh venv, run this once. On Linux with an externally-managed Python
# you may need: pip install --break-system-packages -q ...
!pip install -q transformers accelerate albumentations opencv-python-headless timm ipywidgets kaggle torchmetrics tqdm scikit-image


In [ ]:
import os, sys, glob, json, random, shutil, zipfile, time
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ---- Auto-scaled config based on what hardware you actually have ----
# Edit these by hand if you know your exact VRAM/RAM budget and want to push higher or lower.
if DEVICE.type == "cuda":
    # Fixed input size every step -> let cuDNN autotune the fastest conv algorithms for it.
    torch.backends.cudnn.benchmark = True
    # Ampere+ (RTX 3050 included) supports TF32 matmuls — free speedup, negligible accuracy cost.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    gpu_name = torch.cuda.get_device_properties(0).name
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

    if vram_gb >= 8:
        IMG_SIZE = 192
        BATCH_SIZE = 4
        GRAD_ACCUM_STEPS = 2
        MODEL_NAME = "nvidia/mit-b0"
        EPOCHS = 25
    else:
        # 4GB-class laptop GPUs (RTX 3050 4GB etc.) — mit-b0, 192px, batch 4,
        # gradient-accumulated x2 so the *effective* batch (8) is still stable for BatchNorm/training
        # dynamics, without ever holding more than a batch of 4 in VRAM at once.
        IMG_SIZE = 192
        BATCH_SIZE = 4
        GRAD_ACCUM_STEPS = 2
        MODEL_NAME = "nvidia/mit-b0"
        EPOCHS = 20
        print("Low VRAM (<6GB) detected — mit-b0, 192px, batch 4 (x2 grad accumulation = effective batch 8).")
        print("If you still hit CUDA OOM: drop BATCH_SIZE to 2-3, or IMG_SIZE to 160, and close other GPU apps "
              "(browser tabs, Discord, etc).")

    PIN_MEMORY = True
    # Windows + Jupyter multiprocessing DataLoader workers can hang or be extremely slow — Windows
    # uses 'spawn', which re-imports the whole notebook's __main__ context into every worker process.
    # Keeping this at 0 on Windows trades a little data-loading parallelism for not hanging.
    # On Linux/Mac 'fork' is used instead, which doesn't have this problem, so we use more cores there.
    NUM_WORKERS = 0 if os.name == "nt" else min(6, os.cpu_count() or 2)
else:
    IMG_SIZE = 160
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 1
    MODEL_NAME = "nvidia/mit-b0"   # b2 is not realistic on CPU RAM
    EPOCHS = 8
    PIN_MEMORY = False
    NUM_WORKERS = 0 if os.name == "nt" else 2
    torch.set_num_threads(os.cpu_count())  # make sure PyTorch actually uses all your CPU cores
    print(f"No GPU detected — auto-scaled to IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, mit-b0, "
          f"EPOCHS={EPOCHS}, using all {os.cpu_count()} CPU threads.")
    print("If this still OOMs, close other apps/browser tabs to free RAM, or try IMG_SIZE=128.")
    print("Dataset is also capped at ~1200 train images on CPU (see section 3) to keep an epoch feasible.")

LR = 6e-5

# ---- Checkpointing config (this is what makes training pausable/resumable) ----
CKPT_DIR = "checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
LATEST_CKPT_PATH = os.path.join(CKPT_DIR, "latest.pt")   # overwritten every epoch — used to resume
BEST_CKPT_PATH   = os.path.join(CKPT_DIR, "best.pt")     # overwritten only when val IoU improves — used for inference
RESUME = True   # set to False to ignore any saved checkpoint and start clean from epoch 1

print(f"Config -> model: {MODEL_NAME}, img_size: {IMG_SIZE}, batch_size: {BATCH_SIZE} "
      f"(x{GRAD_ACCUM_STEPS} grad-accum = effective {BATCH_SIZE * GRAD_ACCUM_STEPS}), "
      f"epochs: {EPOCHS}, workers: {NUM_WORKERS}")
print(f"Checkpoints -> latest: {LATEST_CKPT_PATH}   best: {BEST_CKPT_PATH}")


## 2. Get the Datasets

Downloads both datasets straight from Kaggle using your local `kaggle.json` credentials (no Colab upload prompt needed).


In [ ]:
KAGGLE_JSON_PATH = "Kaggle.json"

os.makedirs("data", exist_ok=True)
!kaggle datasets download -d lakshaymiddha/crack-segmentation-dataset -p data/crackseg9k --unzip
!kaggle datasets download -d vangiap/crackvision12k -p data/crackvision12k --unzip

DATASET_ROOTS = ["data/crackseg9k", "data/crackvision12k"]
print("Dataset roots:")
for r in DATASET_ROOTS:
    print(" -", r)


In [ ]:
# Auto-discover (image_dir, mask_dir) pairs inside each dataset root.
# Heuristic: any folder literally named one of the IMAGE_NAMES that has a sibling
# folder named one of the MASK_NAMES, matched by basename count / filename overlap.

IMAGE_DIR_HINTS = {"images", "image", "img", "imgs", "rgb", "train_img", "train_images"}
MASK_DIR_HINTS  = {"masks", "mask", "gt", "groundtruth", "labels", "annotations", "train_lab", "train_masks"}

def find_pairs(root):
    pairs = []
    for dirpath, dirnames, filenames in os.walk(root):
        base = os.path.basename(dirpath).lower()
        if base in IMAGE_DIR_HINTS:
            # look for a sibling mask dir under the same parent
            parent = os.path.dirname(dirpath)
            for sib in os.listdir(parent):
                if sib.lower() in MASK_DIR_HINTS:
                    img_dir = dirpath
                    mask_dir = os.path.join(parent, sib)
                    n_img = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))])
                    if n_img > 0:
                        pairs.append((img_dir, mask_dir))
    return pairs

all_pairs = []
for root in DATASET_ROOTS:
    found = find_pairs(root)
    print(f"{root}: found {len(found)} image/mask folder pair(s)")
    for i, m in found:
        print("   images:", i)
        print("   masks :", m)
    all_pairs.extend(found)

if not all_pairs:
    raise RuntimeError(
        "No image/mask pairs auto-discovered. Print `DATASET_ROOTS` contents with "
        "os.walk and manually set `all_pairs = [(img_dir, mask_dir), ...]` below."
    )

# Manual override example (uncomment & edit if auto-discovery picks the wrong folders):
# all_pairs = [
#     ("/kaggle/input/crack-segmentation-dataset/.../images", "/kaggle/input/crack-segmentation-dataset/.../masks"),
#     ("/kaggle/input/crackvision12k/.../images", "/kaggle/input/crackvision12k/.../masks"),
# ]


## 3. Build a Unified File List

In [ ]:
def match_image_mask_files(img_dir, mask_dir):
    img_files = sorted(glob.glob(os.path.join(img_dir, "*")))
    mask_lookup = {}
    for m in glob.glob(os.path.join(mask_dir, "*")):
        stem = os.path.splitext(os.path.basename(m))[0]
        mask_lookup[stem] = m

    matched = []
    for img_path in img_files:
        stem = os.path.splitext(os.path.basename(img_path))[0]
        # some datasets suffix masks with _mask / _gt
        for cand in (stem, stem + "_mask", stem + "_gt", stem.replace("_img", "")):
            if cand in mask_lookup:
                matched.append((img_path, mask_lookup[cand]))
                break
    return matched

all_samples = []
for img_dir, mask_dir in all_pairs:
    m = match_image_mask_files(img_dir, mask_dir)
    print(f"{img_dir} -> {len(m)} matched pairs")
    all_samples.extend(m)

print("Total matched image/mask pairs across both datasets:", len(all_samples))
random.shuffle(all_samples)

n_val = max(1, int(0.1 * len(all_samples)))
val_samples = all_samples[:n_val]
train_samples = all_samples[n_val:]

# On CPU, training on the full combined dataset (likely 15-20k+ images) is what caused the
# 480min/epoch. Cap the dataset size hard on CPU so an epoch is actually finishable.
# On GPU this stays uncapped (full dataset).
if DEVICE.type != "cuda":
    MAX_TRAIN_SAMPLES = 1200
    MAX_VAL_SAMPLES = 150
    if len(train_samples) > MAX_TRAIN_SAMPLES:
        train_samples = train_samples[:MAX_TRAIN_SAMPLES]
    if len(val_samples) > MAX_VAL_SAMPLES:
        val_samples = val_samples[:MAX_VAL_SAMPLES]
    print(f"CPU detected — capped to {len(train_samples)} train / {len(val_samples)} val samples "
          f"(edit MAX_TRAIN_SAMPLES/MAX_VAL_SAMPLES above to change).")

print("Train:", len(train_samples), "Val:", len(val_samples))


## 4. Dataset & Augmentations

In [ ]:
train_tf_list = [
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.3),
    A.RandomBrightnessContrast(p=0.3),
]
if DEVICE.type == "cuda":
    train_tf_list.append(A.GaussNoise(p=0.15))  # skip on CPU — noticeable per-image cost, not worth it there
train_tf_list += [
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
]
train_tf = A.Compose(train_tf_list)

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class CrackDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.uint8)  # binarize: crack=1, background=0

        augmented = self.transform(image=image, mask=mask)
        image = augmented["image"]
        mask = augmented["mask"].long()
        return image, mask

train_ds = CrackDataset(train_samples, train_tf)
val_ds = CrackDataset(val_samples, val_tf)

# persistent_workers keeps worker processes alive between epochs (skips repeated startup cost);
# prefetch_factor lets each worker queue up batches ahead of time so the GPU is rarely waiting on I/O.
_loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                      persistent_workers=(NUM_WORKERS > 0),
                      prefetch_factor=(4 if NUM_WORKERS > 0 else None))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **_loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **_loader_kwargs)

print("Train batches:", len(train_loader), " Val batches:", len(val_loader))


In [ ]:
# Quick sanity check — visualize a batch
imgs, masks = next(iter(train_loader))
n_show = min(4, imgs.shape[0])  # don't assume batch size >= 4 — it can be 2 or 3 on low-VRAM/CPU configs
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
if n_show == 1:
    axes = axes.reshape(2, 1)  # keep indexing consistent when there's only 1 column
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
for i in range(n_show):
    img = (imgs[i] * std + mean).permute(1,2,0).numpy().clip(0,1)
    axes[0, i].imshow(img); axes[0, i].set_title("image"); axes[0, i].axis("off")
    axes[1, i].imshow(masks[i], cmap="gray"); axes[1, i].set_title("mask"); axes[1, i].axis("off")
plt.tight_layout(); plt.show()


## 5. Model — SegFormer (Vision Transformer for Segmentation)

Using `nvidia/mit-b2` (hierarchical ViT / SegFormer-B2 encoder) from HuggingFace, fine-tuned with a 2-class segmentation head (background / crack). B2 is deeper/wider than B0 — better accuracy, more GPU memory needed.


In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerConfig

id2label = {0: "background", 1: "crack"}
label2id = {v: k for k, v in id2label.items()}

model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
model.to(DEVICE)
print("Model loaded:", MODEL_NAME)


## 6. Loss (CrossEntropy + Dice) & Metrics

In [ ]:
def dice_loss(logits, targets, eps=1e-6):
    probs = F.softmax(logits, dim=1)[:, 1]  # crack channel
    targets = targets.float()
    intersection = (probs * targets).sum(dim=(1,2))
    union = probs.sum(dim=(1,2)) + targets.sum(dim=(1,2))
    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice.mean()

ce_loss_fn = nn.CrossEntropyLoss()

def combined_loss(logits, targets):
    return ce_loss_fn(logits, targets) + dice_loss(logits, targets)

@torch.no_grad()
def iou_score(logits, targets, eps=1e-6):
    preds = torch.argmax(logits, dim=1)
    intersection = ((preds == 1) & (targets == 1)).sum(dim=(1,2)).float()
    union = ((preds == 1) | (targets == 1)).sum(dim=(1,2)).float()
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()


## 7. Training Loop (pausable / resumable)

This cell checkpoints to disk after **every** epoch, so you can stop it whenever you want and
pick up later without losing progress:

- `checkpoints/latest.pt` is overwritten after every epoch — this is what resume reads from.
- `checkpoints/best.pt` is overwritten only when validation IoU improves — this is what you'd load for inference.
- Writes are atomic (write to a temp file, then rename) so an interrupt mid-save can't corrupt the checkpoint file.

**To pause:** just interrupt the kernel (Jupyter's stop/interrupt button) or close Jupyter entirely. The most recent fully-completed epoch is already saved.

**To resume:** re-run this cell (or re-run the whole notebook — no need for a fresh kernel). It reads
`checkpoints/latest.pt`, restores the model/optimizer/scheduler/scaler state, and continues from the next
epoch — e.g. if it was interrupted partway through epoch 6, it'll redo epoch 6 (since epoch 6 hadn't
finished when it was saved) and continue from there, not from epoch 1.

Set `RESUME = False` in the config cell (Section 1) if you ever want to force a clean run instead.

**To test the model without waiting for training to finish**, skip ahead to Section 9 — it loads whatever
checkpoint exists (best if available, otherwise the latest in-progress one) independently of this cell.


In [ ]:
from tqdm.auto import tqdm

ENCODER_LR = 1e-5   # pretrained MiT backbone — smaller updates, don\'t wreck pretrained features
HEAD_LR = 1e-4       # randomly-initialized segmentation head — kept 10x encoder LR since it starts from scratch
WARMUP_FRAC = 0.1    # first 10% of total steps ramp LR up linearly from 0

encoder_params = [p for n, p in model.named_parameters() if "decode_head" not in n]
head_params = [p for n, p in model.named_parameters() if "decode_head" in n]

optimizer = torch.optim.Adam([
    {"params": encoder_params, "lr": ENCODER_LR},
    {"params": head_params, "lr": HEAD_LR},
], weight_decay=1e-4)

optimizer_steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM_STEPS)
total_steps = EPOCHS * optimizer_steps_per_epoch
warmup_steps = int(WARMUP_FRAC * total_steps)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    # cosine decay over the remaining steps, down toward ~0
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
print(f"Optimizer: encoder_lr={ENCODER_LR}, head_lr={HEAD_LR}, warmup_steps={warmup_steps}/{total_steps}")
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

history = {"train_loss": [], "val_loss": [], "val_iou": []}
best_iou = 0.0
start_epoch = 0


def save_checkpoint(path, completed_epoch):
    """Atomic save: write to a temp file then rename, so an interrupt mid-write can\'t corrupt the file."""
    tmp_path = path + ".tmp"
    torch.save({
        "epoch": completed_epoch,           # 0-indexed: last epoch that fully finished
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "history": history,
        "best_iou": best_iou,
        "config": {"MODEL_NAME": MODEL_NAME, "IMG_SIZE": IMG_SIZE, "EPOCHS": EPOCHS},
    }, tmp_path)
    os.replace(tmp_path, path)  # atomic rename on both POSIX and Windows


# ---- Resume from the last completed epoch, if a checkpoint exists ----
if RESUME and os.path.exists(LATEST_CKPT_PATH):
    ckpt = torch.load(LATEST_CKPT_PATH, map_location=DEVICE, weights_only=False)
    if ckpt["config"]["MODEL_NAME"] != MODEL_NAME or ckpt["config"]["IMG_SIZE"] != IMG_SIZE:
        print("Saved checkpoint was trained with a different MODEL_NAME/IMG_SIZE than the current config — "
              "starting fresh instead of resuming (delete checkpoints/ if this is unexpected).")
    else:
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        history = ckpt["history"]
        best_iou = ckpt["best_iou"]
        start_epoch = ckpt["epoch"] + 1
        print(f"Resuming from epoch {start_epoch + 1}/{EPOCHS} (best val IoU so far: {best_iou:.4f})")

if start_epoch >= EPOCHS:
    print(f"Checkpoint already completed all {EPOCHS} epochs. Increase EPOCHS in Section 1 and re-run "
          f"this cell if you want to keep training.")

try:
    for epoch in range(start_epoch, EPOCHS):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", leave=False)
        for step, (imgs, masks) in enumerate(pbar):
            imgs, masks = imgs.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                outputs = model(pixel_values=imgs)
                logits = F.interpolate(outputs.logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
                loss = combined_loss(logits, masks) / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()

            is_last_batch = (step + 1) == len(train_loader)
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or is_last_batch:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()

            running_loss += loss.item() * GRAD_ACCUM_STEPS
            pbar.set_postfix(loss=f"{running_loss / (step + 1):.4f}")
        train_loss = running_loss / len(train_loader)

        model.eval()
        val_loss, val_iou = 0.0, 0.0
        with torch.no_grad():
            for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]", leave=False):
                imgs, masks = imgs.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
                outputs = model(pixel_values=imgs)
                logits = F.interpolate(outputs.logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
                loss = combined_loss(logits, masks)
                val_loss += loss.item()
                val_iou += iou_score(logits, masks)
        val_loss /= len(val_loader)
        val_iou /= len(val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_iou"].append(val_iou)

        print(f"Epoch {epoch+1}/{EPOCHS} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | val_IoU {val_iou:.4f}")

        if val_iou > best_iou:
            best_iou = val_iou
            save_checkpoint(BEST_CKPT_PATH, epoch)
            print(f"  -> new best checkpoint (IoU {best_iou:.4f}) saved to {BEST_CKPT_PATH}")

        # Saved every epoch regardless of whether it was the best — this is what makes resuming possible.
        save_checkpoint(LATEST_CKPT_PATH, epoch)

except KeyboardInterrupt:
    print(f"\nInterrupted during epoch {epoch + 1}. The last fully-completed epoch is safely saved "
          f"in \'{LATEST_CKPT_PATH}\'.")
    print("Just re-run this cell whenever you\'re ready (no need to restart the kernel or re-run earlier "
          "cells) — it will resume from there.")

print("Training loop finished. Best val IoU so far:", best_iou)


## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(history["val_iou"], label="val IoU", color="green")
axes[1].set_title("Validation IoU"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()


## 9. Load a Checkpoint & Visualize Predictions on Validation Set

This cell is **standalone** — it only needs Section 1 (imports/config) and Section 2-4 (data/loaders)
to have been run. You don't need to run the training loop in this same session at all: if you trained
earlier, closed Jupyter, and came back later, this still works as long as `checkpoints/` exists on disk.

It prefers `checkpoints/best.pt` (highest validation IoU seen so far). If training hasn't produced a
"best" checkpoint yet (e.g. you're only a couple epochs in and want to sanity-check progress), it falls
back to `checkpoints/latest.pt` — the most recent epoch, whatever it is.


In [ ]:
from transformers import SegformerForSemanticSegmentation

CHECKPOINT_TO_LOAD = BEST_CKPT_PATH if os.path.exists(BEST_CKPT_PATH) else LATEST_CKPT_PATH

if not os.path.exists(CHECKPOINT_TO_LOAD):
    raise FileNotFoundError(
        f"No checkpoint found at {BEST_CKPT_PATH} or {LATEST_CKPT_PATH} — train at least one epoch first."
    )

ckpt = torch.load(CHECKPOINT_TO_LOAD, map_location=DEVICE, weights_only=False)
which = "best" if CHECKPOINT_TO_LOAD == BEST_CKPT_PATH else "latest (still mid-training)"
print(f"Loading {which} checkpoint: {CHECKPOINT_TO_LOAD} "
      f"(epoch {ckpt['epoch'] + 1}, best val IoU so far {ckpt['best_iou']:.4f})")

model = SegformerForSemanticSegmentation.from_pretrained(
    ckpt["config"]["MODEL_NAME"],
    num_labels=2,
    id2label={0: "background", 1: "crack"},
    label2id={"background": 0, "crack": 1},
    ignore_mismatched_sizes=True,
)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE)
model.eval()
print("Model ready for inference.")

imgs, masks = next(iter(val_loader))
imgs_gpu = imgs.to(DEVICE)
with torch.no_grad():
    outputs = model(pixel_values=imgs_gpu)
    logits = F.interpolate(outputs.logits, size=imgs.shape[-2:], mode="bilinear", align_corners=False)
    preds = torch.argmax(logits, dim=1).cpu()

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

n_show = min(4, imgs.shape[0])
fig, axes = plt.subplots(3, n_show, figsize=(4 * n_show, 12))
if n_show == 1:
    axes = axes.reshape(3, 1)
for i in range(n_show):
    img = (imgs[i] * std + mean).permute(1, 2, 0).numpy().clip(0, 1)
    axes[0, i].imshow(img); axes[0, i].set_title("input"); axes[0, i].axis("off")
    axes[1, i].imshow(masks[i], cmap="gray"); axes[1, i].set_title("ground truth"); axes[1, i].axis("off")
    axes[2, i].imshow(preds[i], cmap="gray"); axes[2, i].set_title("prediction"); axes[2, i].axis("off")
plt.tight_layout(); plt.show()


## 9.5 Full Metrics — Accuracy, Precision, Recall, F1, IoU, Dice

The training loop only tracked loss + IoU per epoch. This cell computes the complete set of
pixel-level metrics on the **entire** train and val sets (not just one batch), aggregated
properly across all pixels rather than averaged per-batch (averaging per-batch is biased
when batches have different amounts of crack pixels).

Note: this notebook only did a 90/10 train/val split — there's no separate held-out test set.
"Train" numbers below will look better than val even on a good model, partly because dropout/
augmentation is off at eval time. If you want a true test set, re-run section 3's split as
train/val/test (e.g. 80/10/10) before training.


In [ ]:
@torch.no_grad()
def evaluate_full_metrics(loader, model, device):
    model.eval()
    tp = fp = fn = tn = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        outputs = model(pixel_values=imgs)
        logits = F.interpolate(outputs.logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
        preds = torch.argmax(logits, dim=1)

        tp += ((preds == 1) & (masks == 1)).sum().item()
        fp += ((preds == 1) & (masks == 0)).sum().item()
        fn += ((preds == 0) & (masks == 1)).sum().item()
        tn += ((preds == 0) & (masks == 0)).sum().item()

    eps = 1e-6
    accuracy  = (tp + tn) / (tp + tn + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1        = 2 * precision * recall / (precision + recall + eps)
    iou       = tp / (tp + fp + fn + eps)
    dice      = 2 * tp / (2 * tp + fp + fn + eps)

    return {
        "pixel_accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "iou": iou,
        "dice": dice,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }

# Use the *non-augmented* transform for the train set here — otherwise "train accuracy"
# reflects performance on flipped/rotated/noised images, not the real training data.
train_eval_ds = CrackDataset(train_samples, val_tf)
train_eval_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print("Evaluating on train set...")
train_metrics = evaluate_full_metrics(train_eval_loader, model, DEVICE)
print("Evaluating on val set...")
val_metrics = evaluate_full_metrics(val_loader, model, DEVICE)

print(f"{'Metric':<16}{'Train':>10}{'Val':>10}")
print("-" * 36)
for key in ["pixel_accuracy", "precision", "recall", "f1_score", "iou", "dice"]:
    print(f"{key:<16}{train_metrics[key]:>10.4f}{val_metrics[key]:>10.4f}")


## 10. Try It Yourself — Upload an Image

Three ways to test, pick whichever works in your local Jupyter setup:

- **Widget upload** (needs `ipywidgets` working in your Jupyter install) — run the cell below and use the upload button.
- **Native file picker** (most reliable in VS Code) — opens a real Windows "choose file" dialog, no widget rendering needed.
- **Local file path** — manually set `TEST_IMAGE_PATH` if you just want to hardcode one image.

**Measurements reported per crack region:** width (mm), length (mm), and overall coverage (%). Each
region is also classified into a severity band:

| Severity | Estimated crack width |
|---|---|
| 🟢 Green | < 1 mm |
| 🟡 Yellow | 1 mm – 3 mm |
| 🔴 Red | > 3 mm |

All mm conversions use the camera's field-of-view and an assumed capture distance (see the "Camera
Working Distance Analysis" section of the project report). Set `CAMERA_DISTANCE_M` below to match your
actual drone-to-wall flight distance.

**How width and length are estimated (approximate, per crack region):**
- **Width** = 2 × the largest inscribed-circle radius (distance transform max) — the thickest point of that crack.
- **Length** = pixel count of the region's morphological skeleton (its 1-pixel-wide centerline) — approximates
  the crack's total path length. Most accurate for reasonably straight, non-branching cracks; can be
  overestimated for blob-like or heavily branching regions.
- **Coverage** = total crack pixels ÷ total image pixels, as a percentage of the whole image.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
from skimage.morphology import skeletonize
import io

infer_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ---- Severity / measurement config ----
# Converts measured pixel widths/lengths to real-world mm using the camera's field of view
# and an assumed capture distance. See the project report's "Camera Working Distance Analysis"
# for the derivation of these numbers (EMEET C950: 70 deg FOV, fixed focus 0.5m-1.2m).
CAMERA_FOV_DEG = 70.0       # horizontal field of view of the inspection camera
CAMERA_DISTANCE_M = 0.5     # assumed drone-to-wall distance at capture time -- match this to your flight setup

GREEN_MAX_MM = 1.0          # crack width below this -> minor (green)
YELLOW_MAX_MM = 3.0         # crack width below this -> moderate (yellow); at/above -> severe (red)

SEVERITY_COLORS = {
    "green":  (0, 200, 0),
    "yellow": (255, 200, 0),
    "red":    (255, 0, 0),
}


def px_to_mm(length_px, image_width_px, distance_m=CAMERA_DISTANCE_M, fov_deg=CAMERA_FOV_DEG):
    """Convert a measured pixel length/width to real-world mm using the camera's FOV-derived ground sampling distance."""
    frame_width_m = 2 * distance_m * np.tan(np.radians(fov_deg / 2))
    mm_per_pixel = (frame_width_m * 1000) / image_width_px
    return length_px * mm_per_pixel


def classify_severity(width_mm):
    if width_mm < GREEN_MAX_MM:
        return "green"
    elif width_mm < YELLOW_MAX_MM:
        return "yellow"
    else:
        return "red"


def estimate_crack_geometry(pred_mask, image_width_px, min_area_px=15):
    """
    For each connected crack region, estimate:
      - width_px:  2 * max(distance transform) -- thickest point of the region
      - length_px: pixel count of the morphological skeleton -- approximate centerline path length
    Both are approximations; most reliable for reasonably straight, non-branching cracks.
    """
    mask_u8 = (pred_mask > 0).astype(np.uint8)
    num_labels, labels = cv2.connectedComponents(mask_u8)
    regions = []
    for label_id in range(1, num_labels):
        comp_mask = (labels == label_id).astype(np.uint8)
        area_px = int(comp_mask.sum())
        if area_px < min_area_px:
            continue  # skip tiny noise specks

        dist = cv2.distanceTransform(comp_mask, cv2.DIST_L2, 5)
        width_px = 2.0 * float(dist.max())

        skeleton = skeletonize(comp_mask.astype(bool))
        length_px = float(skeleton.sum())
        if length_px < 1.0:
            # blob too small/round to skeletonize meaningfully -- fall back to area/width estimate
            length_px = area_px / max(width_px, 1.0)

        width_mm = px_to_mm(width_px, image_width_px)
        length_mm = px_to_mm(length_px, image_width_px)

        regions.append({
            "label": label_id, "mask": comp_mask, "area_px": area_px,
            "width_px": width_px, "width_mm": width_mm,
            "length_px": length_px, "length_mm": length_mm,
            "severity": classify_severity(width_mm),
        })
    return regions


def predict_and_show(image_bytes):
    pil_img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    np_img = np.array(pil_img)
    orig_h, orig_w = np_img.shape[:2]

    augmented = infer_tf(image=np_img)
    tensor_img = augmented["image"].unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        outputs = model(pixel_values=tensor_img)
        logits = F.interpolate(outputs.logits, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        probs = F.softmax(logits, dim=1)[0, 1].cpu().numpy()  # crack probability map
        pred_mask = (probs > 0.5).astype(np.uint8)

    pred_mask_resized = cv2.resize(pred_mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    crack_pixel_pct = 100.0 * pred_mask_resized.sum() / pred_mask_resized.size

    regions = estimate_crack_geometry(pred_mask_resized, orig_w)

    overlay = np_img.copy()
    for r in regions:
        overlay[r["mask"] == 1] = SEVERITY_COLORS[r["severity"]]
    blended = cv2.addWeighted(np_img, 0.6, overlay, 0.4, 0)

    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    axes[0].imshow(np_img); axes[0].set_title("uploaded image"); axes[0].axis("off")
    axes[1].imshow(pred_mask_resized, cmap="gray"); axes[1].set_title("predicted crack mask"); axes[1].axis("off")
    axes[2].imshow(blended); axes[2].set_title(f"severity overlay -- coverage: {crack_pixel_pct:.2f}%"); axes[2].axis("off")
    plt.tight_layout(); plt.show()

    if not regions:
        print(f"Verdict: No significant crack detected (crack pixel coverage: {crack_pixel_pct:.2f}%)")
        return

    severity_counts = {"green": 0, "yellow": 0, "red": 0}
    total_length_mm = 0.0
    max_width_mm = 0.0
    for r in regions:
        severity_counts[r["severity"]] += 1
        total_length_mm += r["length_mm"]
        max_width_mm = max(max_width_mm, r["width_mm"])
    overall = "red" if severity_counts["red"] > 0 else ("yellow" if severity_counts["yellow"] > 0 else "green")

    print(f"Verdict: CRACK DETECTED -- overall severity: {overall.upper()}")
    print(f"  Coverage: {crack_pixel_pct:.2f}% of image   |   Total crack length: {total_length_mm:.1f} mm   |   "
          f"Widest crack: {max_width_mm:.2f} mm   (assuming {CAMERA_DISTANCE_M} m capture distance)")
    print()
    print(f"  {'Region':<8}{'Width (mm)':<14}{'Length (mm)':<15}{'Severity':<10}")
    print("  " + "-" * 47)
    for i, r in enumerate(regions, start=1):
        print(f"  {i:<8}{r['width_mm']:<14.2f}{r['length_mm']:<15.1f}{r['severity']:<10}")


uploader = widgets.FileUpload(accept="image/*", multiple=False, description="Upload image")
output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output(wait=True)
        if uploader.value:
            # ipywidgets>=8 stores value as tuple of dicts; older versions as dict
            item = list(uploader.value.values())[0] if isinstance(uploader.value, dict) else uploader.value[0]
            content = item["content"] if isinstance(item, dict) else item.content
            predict_and_show(bytes(content))

uploader.observe(on_upload_change, names="value")
display(uploader, output)


In [ ]:
# --- Fallback: test with a local file path instead of the widget above ---
TEST_IMAGE_PATH = "path/to/your/image.jpg"  # <-- set this to an actual file on your machine

if os.path.exists(TEST_IMAGE_PATH):
    with open(TEST_IMAGE_PATH, "rb") as f:
        predict_and_show(f.read())
else:
    print(f"Set TEST_IMAGE_PATH to a real file to use this cell (currently: {TEST_IMAGE_PATH})")


In [ ]:
# --- Alternative: native OS file picker (use this if the ipywidgets button above
# doesn't render or doesn't respond — common in VS Code Jupyter) ---
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()          # hide the empty root window
root.attributes("-topmost", True)  # bring the dialog to the front

picked_path = filedialog.askopenfilename(
    title="Select an image to test",
    filetypes=[("Images", "*.jpg *.jpeg *.png *.bmp")]
)
root.destroy()

if picked_path:
    print("Selected:", picked_path)
    with open(picked_path, "rb") as f:
        predict_and_show(f.read())
else:
    print("No file selected.")
